# LangChain으로 만드는 웹 기반 챗봇 실습  

##  1. 과정 소개 및 개요
- **학습 목표**
  - AI 챗봇의 기본 개념과 발전 과정을 이해한다.
  - LangChain, Streamlit, FAISS 등 주요 기술의 역할과 장점을 이해한다.
  - LCEL을 사용하여 확장성 있는 체인을 직접 설계할 수 있다.
  - **웹 기반으로 자신이 커스텀한 챗봇을 배포할 수 있는 역량을 갖춘다.**

- **핵심 키워드**
  - LLM Chatbot, LangChain, LCEL, Prompt, Streamlit, FAISS, RAG  

### 환경 설정 

#### (1) 필수 라이브러리 설치 
웹 페이지(UI) 를 직접 띄워야 하기 때문에, Google Colab보다는  
본인 PC의 로컬 환경(VSCode, Terminal 등) 에서 실행하는 것을 권장합니다.

먼저 실습에 필요한 주요 패키지를 설치합니다.
터미널 또는 VSCode의 터미널 창에서 다음 명령어를 입력하세요.
```bash
pip install -U langchain langchain-openai streamlit faiss-cpu python-dotenv tiktoken
```


#### (2) 라이브러리별 역할 설명

| 라이브러리                | 주요 역할        | 설명                                                                                                  |
| -------------------- | ------------ | --------------------------------------------------------------------------------------------------- |
| **langchain**        | 체인 프레임워크 핵심  | LLM(대형언어모델)을 중심으로 다양한 입력·출력·도구를 연결하는 프레임워크. `PromptTemplate`, `LLMChain`, `Retriever`, `LCEL` 등을 제공 |
| **langchain-openai** | OpenAI 연동 모듈 | OpenAI의 GPT 모델(`ChatOpenAI`)을 LangChain 체인 안에서 바로 사용할 수 있도록 연결해줌                                    |
| **streamlit**        | 웹 인터페이스 구현   | Python 코드 몇 줄로 대화형 웹 UI(챗봇 화면, 입력창, 출력창 등)를 만들 수 있는 간단한 프레임워크                                       |
| **faiss-cpu**        | 벡터 검색 DB     | 문서 검색 기반 챗봇(RAG) 구현 시, 텍스트 임베딩을 벡터로 저장하고 유사한 내용을 빠르게 검색하는 역할                                        |
| **python-dotenv**    | 환경변수 관리      | `.env` 파일에서 OpenAI API Key 등 민감한 정보를 안전하게 불러와 코드에 직접 노출하지 않게 해줌                                     |

#### (3) OpenAI API 키 설정 (.env 파일, 보안 주의)

**API란?**

하나의 프로그램이 다른 프로그램이나 시스템과 **데이터를 주고받고 기능을 활용할 수 있도록 연결해주는 인터페이스**를 의미합니다.  
즉, **서로 다른 소프트웨어 간의 다리 역할을 하는 통신 규칙**입니다.

예를 들어, OpenAI의 **GPT API**는 사용자가 직접 모델을 설치하지 않아도  
OpenAI 서버에 **질문(요청)** 을 보내고, 그 결과로 **답변(응답)** 을 받을 수 있게 해주는 API입니다.

→ 쉽게 말해, **필요할 때마다 GPT에게 요청을 보내 답변을 받아오는 온라인 시스템**이라고 생각하시면 됩니다.

※ 이 API는 **사용량 기반으로 과금되는 유료 서비스**이며, 웹에서 제공되는  
ChatGPT의 무료·유료 플랜과는 **별도의 요금 체계**로 운영됩니다.


※ 현재 사용 가능한 대표 GPT API 모델 일부 (2026년 3월 기준)

| 모델 이름 | 설명 | 입력 가격 (per 1M tokens) | 출력 가격 (per 1M tokens) | 추천 용도 |
|-----------|--------|---------------------------:|----------------------------:|-----------|
| `gpt-3.5-turbo` | 초창기 레거시 모델 | **$0.5** | **$1.5** | 기존 예제 호환, 구형 자료 참고용 |
| `gpt-4o-mini` | 빠르고 저렴한 범용 소형 모델 | **$0.30** | **$1.20** | 가벼운 멀티모달 앱, FAQ 챗봇, 분류/추출, 파인튜닝·증류용 |
| `gpt-4.1-mini` | 성능, 속도, 비용 측면에서 균형이 좋은 모델 | **$0.8** | **$3.2** | 지시 이행, 툴 호출, 긴 컨텍스트 처리, 실무형 API 앱 |
| `gpt-4.1-nano` | 더 가볍고 저렴한 초경량 모델 | **$0.2** | **$0.8** | 초저비용 분류, 간단한 자동화, 대량 처리, 저지연 서비스 |
| `o4-mini` | 빠른 추론에 강한 모델 | **$4.0** | **$16.0** | 복잡한 추론, 코딩 보조, 시각 이해, 다단계 문제 해결 |
| `gpt-5-mini` | 빠르고 효율적인 GPT-5 계열 모델 | **$0.25** | **$2.0** | 저지연·고볼륨 실서비스, 명확한 업무형 프롬프트, 범용 실습 |
| `gpt-5-nano` | GPT-5 계열 최저가 모델 | **$0.05** | **$0.4** | 요약, 분류, 태깅, 대량 배치 처리, 입문 실습/초저비용 API |

※ OpenAI는 지속적으로 모델명을 업데이트하고 있으므로, [공식 문서](https://platform.openai.com/docs/models)에서 최신 목록을 확인하는 것이 좋습니다.

최신 요금은 OpenAI 공식 문서에서 확인하세요:  
https://openai.com/api/pricing/

**API Key란?**  
GPT API를 사용하기 위해서는 OpenAI 서버가 요청을 보낸 사용자를 식별하고,  
해당 사용자의 **이용량과 과금 내역을 추적**할 수 있어야 합니다.  
이를 위해 OpenAI는 각 사용자에게 **고유한 인증 토큰(API Key)** 을 발급합니다.

이 키는 **요금 청구의 기준이 되는 중요 정보**이며, 외부에 노출될 경우 **타인이 임의로  
사용해 요금이 발생할 수 있으므로 철저한 보안 관리가 필요**합니다.


**발급 절차**

1. [https://platform.openai.com/](https://platform.openai.com/) 접속  
2. 로그인 또는 계정 신규 생성  
3. 우측 상단 프로필 아이콘 클릭 → `View API Keys` 선택  
4. 결제 수단(신용카드 등) 등록  
5. `+ Create new secret key` 클릭 → **발급된 Key를 복사해 안전한 위치에 보관**

**발급된 Key는 한 번만 표시되므로**, 반드시 즉시 메모장이나 `.env` 파일 등에 백업해 두세요.

실습 폴더 루트에 .env 파일을 생성하고, 강사가 안내한 OpenAI API Key를 아래 형식으로 저장합니다. 
```bash
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
``` 

#### ⚠️ 주의
.env 파일은 절대 깃허브(GitHub)나 외부 저장소에 업로드하지 마세요.  
사용은 `.env` 파일에 저장 후 `python-dotenv`를 이용해 불러오는 방식을 권장합니다.
API Key가 노출되면 과금 위험이 있습니다.

```python
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
```

##  2. LangChain 소개
### 🔹 LangChain이란?

<img src ="image/LangChain.png" width="500">

LangChain은 **대규모 언어모델(LLM)** 을 더 쉽게 활용할 수 있도록 도와주는 **프레임워크**입니다.  

단순히 LLM에게 질문을 던지는 수준을 넘어서, **프롬프트 관리(prompt management)**,  
**메모리(memory)**, **외부 도구(tool) 연동**, **문서 검색(RAG)** 등을 체계적으로 구성할 수 있습니다.

예를 들어, 우리가 ChatGPT 같은 LLM을 사용할 때  
단순히 모델 하나만으로 뛰어난 답변이 만들어지는 것은 아닙니다.  

실제로는 **검색(Search)**, **분석(Analysis)**, **정보 정리(Summarization)** 등  
여러 절차가 백그라운드에서 함께 작동해야 더 정확하고 풍부한 답변을 얻을 수 있습니다.  

LangChain은 이러한 여러 단계를 **하나의 체계적인 구조로 묶어주는 프레임워크**입니다.  
즉, LLM이 단독으로 모든 일을 처리하는 대신  
“사용자 입력 → 관련 정보 검색 → 분석 및 요약 → 결과 생성”과 같은  
복잡한 흐름을 하나의 **체인(chain)** 으로 자동화할 수 있습니다.  

<img src="image/chatgpt.jpg" width="600">

예를 들어, 사용자가 “3박 4일, 30만 원 예산, 맛집 중심의 서울 여행”을 요청하면,  
LangChain은 내부적으로 다음과 같은 과정을 수행합니다.  

> 입력 분석 → 예산과 일정 파악 → 추천 장소 검색 → 일정표 생성 → 자연스러운 문장으로 요약  

이처럼 LangChain은 단순한 대화 요청을 넘어 **검색, 분석, 조합, 요약** 등의 단계를 유기적으로 연결해  
LLM이 더 똑똑하게 작동하도록 돕습니다.  

또한 LangChain은 **LCEL (LangChain Expression Language)** 이라는 표현 언어를 통해  
이러한 여러 단계(components)를 **직관적으로 연결**할 수 있게 해줍니다.  
즉, “LLM → 요약기 → 출력 포맷터”처럼 **함수형 파이프라인**을 선언적으로 작성할 수 있어  
확장성과 유지보수성이 뛰어납니다.

LCEL 예시: 
```text
사용자입력
  | 분석기(요청에서 날짜·예산·키워드 추출)
  | 검색기(분석 결과를 바탕으로 추천 장소 탐색)
  | 일정생성기(추천 장소로 일자별 일정 구성)
  | 요약기(자연스러운 문장으로 결과 정리)
  | 출력
```

이처럼 LCEL은 각 단계를 함수처럼 “연결(pipe)”하여 데이터가 흘러가는 경로를 선언적으로 표현합니다.  
복잡한 제어문 없이 “입력 → 처리 → 출력” 구조를 직관적으로 구성할 수 있어,
체인 전체의 논리 흐름을 한눈에 파악할 수 있습니다.

### 🔹 간단한 예시 코드

In [3]:
# OpenAI API를 LangChain에서 쉽게 쓸 수 있도록 해주는 래퍼 모듈
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import PromptTemplate # 문자열 템플릿 기반으로 프롬프트를 자동 구성해주는 유틸
from langchain_core.output_parsers import StrOutputParser # LLM의 응답 객체를 문자열로 변환해주는 파서 (출력 후처리 단계)
from dotenv import load_dotenv # .env 파일의 환경변수를 자동으로 불러오기 위한 모듈
load_dotenv()  # 실행 시 .env 파일을 찾아 변수들을 환경에 로드

# LCEL 방식: 파이프라인 형태로 선언
llm = ChatOpenAI(model="gpt-5-nano")
prompt = PromptTemplate.from_template("'{topic}' 주제에 대해 한 문장으로 설명해줘.")
output_str = StrOutputParser() # 결과 객체에서 텍스트(content)만 깔끔하게 추출해 문자열로 변환함

# LCEL 표현 (프롬프트 → LLM → 출력)
chain = prompt | llm | output_str
# 준비한 세 가지 부품을 파이프(|) 기호로 한 줄로 연결해 chain(작업 흐름)을 만듭니다.
# 작동 순서: 질문 양식(prompt) 완성 → AI 모델(llm)에게 전달 → 문자열 변환기(output_str)로 답변 정리.

# chain.invoke() 는 전체 파이프라인을 한 번 실행하는 메서드
result = chain.invoke({"topic": "LangChain"})
print(result)

LangChain은 대형 언어 모델(LLM)과 도구, 데이터 소스, 메모리 등을 체인처럼 연결해 복잡한 LLM 기반 애플리케이션을 쉽고 유연하게 설계·구현할 수 있게 해 주는 프레임워크다.


| 항목            | 설명                              |
| ------------- | ------------------------------- |
| **LangChain** | LLM을 활용하기 위한 파이썬 기반 프레임워크       |
| **핵심 기능**     | 프롬프트 관리, 메모리, 도구 연동, 문서 검색(RAG) |
| **LCEL**      | 체인 구성 과정을 간결하게 표현하는 언어          |
| **활용 예시**     | 질문응답, 요약, 문서검색, 챗봇, 자동화된 지식시스템  |


LangChain은 LLM의 “두뇌”를 실무에서 “자동화 가능한 시스템”으로 바꿔주는 연결 프레임워크입니다.  
단순한 모델 호출이 아니라, 프로세스를 설계하고 실행하는 엔진이라고 이해하면 됩니다.

## 3. 프롬프트(Prompt)

**프롬프트(Prompt)** 는 LLM(대형언어모델)에게 “무엇을, 어떻게 해달라”고 **지시하는 문장**입니다.

사람에게 질문할 때도 요청을 명확히 해야 원하는 답을 얻을 수 있듯,  LLM에게도 프롬프트가 곧 **“명령어이자 대화의 시작점”** 이 됩니다.  

예를 들어,  
> 1. “고양이에 대해 설명해줘.”  
> 2. “고양이를 5살 아이가 이해할 수 있게 설명해줘.”  

두 문장은 같은 주제지만 결과는 완전히 다릅니다.  
👉 따라서 **프롬프트의 설계(=프롬프트 엔지니어링)** 가 답변의 질을 결정합니다.

#### LLM은 세 가지 종류의 프롬프트로 대화한다
LLM은 단순히 한 문장만 보고 답하지 않습니다.  
대화의 **맥락(context)** 을 유지하기 위해 다음 세 가지 프롬프트를 함께 사용합니다.

| 종류 | 역할 | 예시 |
|------|------|------|
| **System Prompt** | 모델의 성격과 역할을 설정 | “너는 친절하고 이해심 많은 선생님이야.” |
| **User Prompt** | 사용자의 실제 질문이나 요청 | “인공지능이 뭐야?” |
| **Assistant Prompt** | 모델이 이전에 했던 응답 | “인공지능은 사람의 학습 능력을 모방한 기술이야.” |

#### 왜 이렇게 구분할까?
- **System Prompt** → 모델의 **기본 성격**을 결정합니다. (친절한 선생님 / 냉정한 분석가 등)  
- **User Prompt** → 사용자가 **무엇을 알고 싶은지**를 알려줍니다.  
- **Assistant Prompt** → 이전 대화 내용을 **기억하고 맥락을 유지**하게 도와줍니다.  

이 세 가지를 함께 쓰면 모델은  
> “지금 나는 친절한 선생님이고, 방금 ‘인공지능이 뭐냐’는 질문을 들었구나.”  
라고 이해한 뒤 더 자연스럽고 일관된 답변을 합니다.

### 🔹 LangChain에서의 프롬프트: `PromptTemplate`
LangChain에서는 프롬프트를 **하드코딩(직접 작성)** 하는 대신, **`PromptTemplate`** 클래스를 사용해 “템플릿” 형태로 관리합니다.

#### 예시 1

In [4]:
from langchain_core.prompts  import PromptTemplate

# {topic} 부분에 사용자의 입력이 들어가도록 템플릿 생성
prompt = PromptTemplate.from_template("'{topic}'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘.")

# 실제 실행 시:
prompt.format(topic="LangChain")
# 결과 → "'LangChain'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘."

"'LangChain'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘."

#### 예시 2

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1.  모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 2️. 프롬프트 구성
# - System: 모델의 기본 역할(여행 전문가)
# - User: 사용자의 첫 질문 (가을 여행지 추천)
# - Assistant: 직전에 모델이 대답했던 내용 (AI의 응답)
# - User: 현재 사용자의 새로운 질문 (부산 일정표 요청)
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 여행 전문가야. 사용자의 취향에 맞게 국내 여행지를 추천하고, 일정도 구체적으로 제안해줘."),
    ("user", "가을에 2박 3일로 갈만한 국내 여행지를 추천해줘."),
    ("assistant", "부산, 여수, 속초가 좋을 것 같아요! 각각 바다와 음식, 분위기가 다르니 선택에 따라 즐길 거리가 달라요."),
    ("user", "그럼 이번엔 부산으로 2박 3일 일정표를 만들어줘.")
])

# 3️. 출력 파서 (응답 텍스트만 추출)
parser = StrOutputParser()

# 4️. LCEL 파이프라인 구성 (Prompt → LLM → Parser)
chain = prompt | llm | parser

# 5️. 실행
result = chain.invoke({})

print(result)

좋아요! 가을 분위기를 만끽하는 부산 2박 3일 일정표를 제안드려요. 해운대를 베이스로 도심의 맛집과 바다 풍경을 골고루 즐길 수 있도록 구성했습니다.

숙소 베이스 추천
- 해운대: 바다 전망과 편의시설이 좋고, 하루 코스를 짜기에도 편해요.
- 서면/광복로: 도보와 대중교통 접근이 좋아 이동이 수월합니다. 취향에 맞춰 선택하세요.

2박 3일 일정표 (가을 분위기 추천)

Day 1 - 감천문화마을 + 자갈치시장 / BIFF 광장 + 광안리 야경
- 12:00 도착 및 체크인(해운대 기준)
- 13:00 점심: 자갈치시장 또는 BIFF 광장 근처에서 신선한 해산물 코스나 해물파전
- 14:30 감천문화마을: 알록달록 색감이 아름다운 골목길 산책. 포토 스팟 많음(약 1.5~2시간)
- 16:30 남포동/BIFF 광장 산책: 영화의거리 분위기로 산책하며 간식씩 즐기기
- 18:30 저녁: 해운대/남포동 인근에서 해산물 또는 부산 유명 식당 체험
- 20:30 광안리 해변으로 이동: 광안대교 야경 감상 및 해변 산책
- 22:00 숙소 복귀 및 휴식

Day 2 - 해동용궁사 + 태종대 공원 + Dongbaek/Nurimaru
- 08:00 숙소에서 가벼운 아침
- 09:00 해동용궁사 방문: 바다를 따라 위치한 아름다운 사찰. 대중교통으로 약 60~70분 소요되며 택시 이용 시 편합니다
- 12:00 점심: 해운대/송정 근처의 해물 맛집 또는 해산물 전문점
- 13:30 태종대 공원 이동 및 산책: 전망대와 해안 산책로 따라 걷기(약 2시간)
- 15:30 태종대에서 청사포 방향의 산책 겸 드라이브 코스 추천
- 17:00 동백섬 Dongbaek Park + Nurimaru APEC House 방문: 바다와 벚나무 가로수의 조화로운 풍경
- 19:00 저녁: 해운대 수변공원 인근에서 회/해산물 또는 로컬 음식점
- 21:00 Spa Day Option: Spa Land(센텀시티) 또는 호텔 내 휴식

Day 3 - 전통과 현대의 만남 + 마지막 쇼핑
- 08

#### 예시 3. 퓨샷 러닝(Few-shot Learning)

**Few-shot Learning**은 LLM에게 **“예시 몇 개를 직접 보여줘서 학습 방향을 잡아주는 방법”** 입니다.

AI에게 “이렇게 대답해!”라고 명령하는 것만으로는 부족할 때,  
실제 **입출력 예시를 몇 개 제시해주면** 모델이 그 **패턴과 말투, 구조**를 빠르게 학습해 유사한 답변을 생성합니다.

예를 들어  
> “백종원 대표의 충청도 사투리로 요리 팁을 말해줘.”  
라고만 하면 AI는 사투리의 뉘앙스를 정확히 몰라 평범한 답변을 할 수 있습니다.  

하지만 아래처럼 실제 대화 예시를 1~2개 보여주면  
> “아~ 이런 식으로 말하면 되는구나!”  
하고 패턴을 스스로 학습해 자연스러운 말투를 재현합니다.

**퓨샷 러닝의 장점**

| 장점 | 설명 |
|------|------|
|**말투·스타일 학습 가능** | 특정 인물(예: 백종원, 아이유, 선생님 등)의 말투나 표현방식 재현 |
|**작업 포맷 유지** | 요약, 번역, 해설 등 특정 형식을 예시로 제시하면 그 구조를 그대로 따름 |
|**추가 학습(파인튜닝) 없이 가능** | 단순히 프롬프트 안에 예시 몇 개 넣는 것만으로도 “즉석 학습” 효과 |
|**창의적 제어** | “~처럼 말해줘”, “이런 형식으로 답해줘” 등 자유로운 응용 가능 |


In [ ]:
### 코드 예시: 백종원 말투 흉내내기 (Few-shot Learning)
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# temperature를 약간 높이면(0.8) 표현이 더 자유롭고 창의적이게 됨
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.8)
# - llm = ChatOpenAI(model="gpt-5-mini" 모델을 상위로 높여서 답변 퀄리티 높이자, temperature=0 으로 세팅하면 요청한대로 답변)

# - System: 모델의 기본 역할 지정
# - 예시 대화(few-shot examples): 실제 말투 패턴을 보여줌
# - User: 새로운 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 요리 전문가이자, 백종원 대표의 말투를 자연스럽게 흉내내는 AI야. 충청도 사투리와 친근한 말투로 대답해."),
    # - ("system", "너는 요리 전문가이자, 백종원 대표의 말투를 자연스럽게 흉내내는 AI야. 충청도 사투리와 친근한 말투로 대답해. ~여 혹은 ~잉 으로 대답 끝을 만들어."),
    
    # [예시 1] — 사용자가 물어봤고, 백종원 말투로 답변한 사례
    ("user", "라면 맛있게 끓이는 비법 알려주세요."),
    ("assistant", "그라믄 말이여~ 라면은 물 끓일 때 스프를 같이 넣는 게 포인트여. 그래야 국물이 진~해진다잉."),
    
    # [예시 2] — 또 다른 백종원식 대답 예시
    ("user", "김치찌개 맛있게 끓이는 법은요?"),
    ("assistant", "김치찌개는 묵은지 써야혀~ 그래야 깊은 맛이 나. 돼지고기 좀 넣어주면 금상첨화여잉."),
    
    # [새로운 요청] — 실행 시 전달될 질문 변수. 이제 모델이 같은 말투로 이어서 대답해야 함
    ("user", "{question}")])

# 출력 파서 (응답 텍스트만 추출)
parser = StrOutputParser()

# LCEL 파이프라인 구성 (Prompt → LLM → Parser)
chain = prompt | llm | parser

# 실행 — invoke()에 직접 새로운 질문 전달
result = chain.invoke({"question": "김치전은 어떻게 만들면 맛있어요?"})

print(result)

좋아라~ 김치전 맛있게 만드는 비법도 쉬워요. 이래 하면 바삭하게, 김치 맛도 진하게 나오더라 그려.

재료
- 잘 익은 김치 2컵 정도(썰어서)
- 부침가루 1컵(또는 밀가루+전분 2:1 비율) + 물 1/2컵 정도
- 김치 국물 조금(선택사항, 농도 맞추려 넣기)
- 대파 1대, 송송 썰기
- 달걀 1개(선택, 더 촉촉한 식감 원하면)
- 돼지고기 얇게 썬 것 50g 정도(원하면 넣어도 맛있음)
- 식용유

만드는 법
1) 김치를 잘게 다져서 큰 그릇에 담고, 김치 국물 조금과 반죽 재료를 넣어 반죽을 만든다. 반죽이 너무 묽지 않게, 떠먹는 정도로 맞춰라.
2) 대파와 고기(또는 생략 가능)를 넣고 고루 섞는다. 달걀 넣으면 더 촉촉하고 바삭한 식감이 살아난다.
3) 팬 달궈서 기름 넉넉히 둘러 중불로 낮추고, 반죽을 한 국자 떠서 얇고 넓게 펴준다. 얇게 펴야 바삭하게 구워진다.
4) 한 면이 노릇하게 바삭해지면 뒤집어 반대편도 잘 구워준다. 양면 다 노릇하고 바삭하면 접시에 옮기기.

팁
- 김치의 수분이 많으면 반죽에 물을 조금 덜 넣고, 김치 국물을 조금 남겨두면 깊은 풍미가 난다.
- 더 바삭하게 원하면 반죽에 전분(전분 1/4컵 정도)이나 찰진 느낌의 밀가루를 조금 더 첨가해봐라.
- 팬 온도는 중불 정도로 유지해 타지 않게 한다.
- 찍먹 소스: 간장 2큰술 + 식초 1큰술 + 설탕 1작은술 + 물 조금 + 다진 마늘 약간 + 깨 조금.

바로 해먹으면 맛이 확 살아나니, 뜨거울 때 먹으면 제일 맛있다요. 필요하면 2인분 기준으로 재료 비율도 맞춰줄게요!


##  4. Q&A 챗봇 실습 (LCEL)

앞서 배운 **프롬프트 설계**와 **LCEL 파이프라인 구성법**을 직접 활용하여  
“질문 → 답변” 형태의 **Q&A 챗봇**을 완성해보세요.

단순히 모델을 호출하는 것이 아니라, **프롬프트 구조를 설계**하고 **출력 파서를 연결**하며  
 **LCEL 문법(| 연산자)** 을 이용해 **체인형 파이프라인**을 구성해야 합니다.

##### 실습 시나리오
> 당신은 **AI Q&A 챗봇 개발자**입니다.  
> 사용자가 “기술 관련 질문”을 하면, 챗봇은 **요약 + 예시 + 비유 설명**을 포함한 답변을 하도록 만들어야 합니다.  
> (예: “REST API란?”, “클라우드 컴퓨팅이 뭐야?”, “RAG는 어떤 구조야?” 등)

즉, 단순한 정의를 넘어 “무엇인지 + 왜 중요한지 + 예시” 를 한 번에 설명하는 챗봇을 설계하세요.

**과제 요구사항**

1. **`ChatPromptTemplate`** 을 사용해 아래 구조의 프롬프트를 직접 설계하세요.
   - `system`: 챗봇의 역할 정의  
     (예: “너는 기술 개념을 쉬운 비유와 예시를 섞어서 설명하는 AI 선생님이야.”)
   - `user`: 사용자의 질문을 `{question}` 변수로 전달

2. **`ChatOpenAI`** 모델을 연결하세요.  
   - 모델: `"gpt-5-nano"`  
   - `temperature=0.7` (조금 창의적인 답변 유도)

3. **`StrOutputParser`** 를 사용하여 결과를 문자열로 출력하세요.

4. **LCEL 문법**으로 체인을 완성하세요.  
   예:  
   ```python
   chain = prompt | llm | parser
   ```
5. 실행해보고 답변이 지나치게 딱딱하거나 비유나 예시가 부족하면 Prompt를 직접 수정하면서 성능을 조정해보세요.

**질문 답변 예시:**  

질문 : REST API란?  

답변 :   

① **정의**: REST API(Representational State Transfer Application Programming Interface)는 웹 상에서 서로 다른 프로그램이나 시스템이 데이터를 주고받을 수 있도록 하는 규칙과 방법을 정의한 인터페이스입니다. REST는 HTTP 프로토콜을 기반으로 하며, 자원(Resource)을 URI(Uniform Resource Identifier)로 식별하고, 다양한 HTTP 메서드(예: GET, POST, PUT, DELETE)를 사용하여 자원에 대한 작업을 수행합니다.

② **이유(중요성)**: REST API는 서로 다른 시스템 간의 통신을 쉽고 효율적으로 만들어줍니다. 이를 통해 개발자들은 복잡한 시스템을 간단하게 연결하고, 데이터를 주고받을 수 있습니다. REST API는 웹 기반 애플리케이션에서 표준으로 자리잡고 있어, 다양한 서비스를 통합하고 확장하는 데 매우 중요합니다.

③ **쉬운 예시**: REST API를 설명하기 위해 레스토랑을 비유로 들어볼 수 있습니다. 레스토랑의 메뉴판은 고객이 어떤 음식을 주문할 수 있는지를 보여주는 역할을 합니다. 고객(클라이언트)은 메뉴판(URI)을 보고 원하는 음식을 선택하고, 주문(HTTP 메서드)을 합니다. 주방(서버)은 고객의 주문을 받아 음식을 준비하고, 다시 고객에게 서빙합니다. 이 과정에서 메뉴판은 고객과 주방 간의 소통을 원활하게 해주는 역할을 하며, REST API는 시스템 간의 소통을 원활하게 해주는 역할을 합니다.


In [20]:
### 여기에 과제 요구 사항을 작성하세요
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser 

# temperature를 약간 높이면(0.8) 표현이 더 자유롭고 창의적이게 됨
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7, max_tokens=3000)

# - System: 모델의 기본 역할 지정
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 전공 대학생들을 대상으로 교육하는 기술 개념을 쉬운 비유와 예시를 섞어서 잘 기본 개념을 설명하는 AI 전문교육 강사야. 요약 + 예시 + 비유 설명을 포함하여 간략하게 순번을 달아서 정리한 답변을 해야해"),
    
    # [예시 1] — REST API에 대해 정의 + 이유 + 쉬운 예시로 답변한 사례
    ("user", "REST API란?"),
    ("assistant", "질문 : '{question}'  \n 답변 : \n ① **정의**: REST API(Representational State Transfer Application Programming Interface)는 웹 상에서 서로 다른 프로그램이나 시스템이 데이터를 주고받을 수 있도록 하는 규칙과 방법을 정의한 인터페이스입니다. REST는 HTTP 프로토콜을 기반으로 하며, 자원(Resource)을 URI(Uniform Resource Identifier)로 식별하고, 다양한 HTTP 메서드(예: GET, POST, PUT, DELETE)를 사용하여 자원에 대한 작업을 수행합니다.\n② **이유(중요성)**: REST API는 서로 다른 시스템 간의 통신을 쉽고 효율적으로 만들어줍니다. 이를 통해 개발자들은 복잡한 시스템을 간단하게 연결하고, 데이터를 주고받을 수 있습니다. REST API는 웹 기반 애플리케이션에서 표준으로 자리잡고 있어, 다양한 서비스를 통합하고 확장하는 데 매우 중요합니다.\n③ **쉬운 예시**: REST API를 설명하기 위해 레스토랑을 비유로 들어볼 수 있습니다. 레스토랑의 메뉴판은 고객이 어떤 음식을 주문할 수 있는지를 보여주는 역할을 합니다. 고객(클라이언트)은 메뉴판(URI)을 보고 원하는 음식을 선택하고, 주문(HTTP 메서드)을 합니다. 주방(서버)은 고객의 주문을 받아 음식을 준비하고, 다시 고객에게 서빙합니다. 이 과정에서 메뉴판은 고객과 주방 간의 소통을 원활하게 해주는 역할을 하며, REST API는 시스템 간의 소통을 원활하게 해주는 역할을 합니다."),

    # [예시 2] — 클라우드 컴퓨팅에 대해 정의 + 이유 + 쉬운 예시로 답변한 사례
    ("user", "클라우드 컴퓨팅이 뭐야?"),
    ("assistant", "질문 : '{question}'  \n 답변 : \n ① **정의**: 클라우드 컴퓨팅(Cloud Computing)은 사용자가 서버, 스토리지, 데이터베이스, 소프트웨어 등의 IT 자원을 직접 구매해 물리적으로 구축하지 않고, 인터넷을 통해 필요한 만큼 빌려 쓰고 사용한 만큼 비용을 지불하는 컴퓨팅 서비스입니다.\n② **이유(중요성)**: 초기 서버 구축 비용과 유지보수 부담을 크게 줄일 수 있으며, 트래픽 변화에 따라 서버 자원을 유연하게 늘리거나 줄일 수 있습니다. 또한 인터넷만 연결되어 있다면 언제 어디서나 동일한 작업 환경과 데이터에 접근할 수 있어 현대 IT 서비스의 필수 인프라로 자리 잡았습니다.\n③ **쉬운 예시**: 클라우드 컴퓨팅은 가정에서 '전기를 끌어다 쓰는 방식'에 비유할 수 있습니다. 우리가 집에서 전기를 쓰기 위해 마당에 직접 발전소를 짓지 않고 한국전력(클라우드 제공사)에서 전기를 공급받아 사용한 만큼 전기세를 내는 것과 같습니다. 마찬가지로 기업이나 개인도 직접 서버실을 만드는 대신 AWS나 구글 드라이브 같은 클라우드 서비스를 통해 필요한 저장 공간과 연산 능력만 빌려서 사용합니다."),

    # [예시 3] — RAG에 대해 정의 + 이유 + 쉬운 예시로 답변한 사례
    ("user", "RAG는 어떤 구조야?"),
    ("assistant", "질문 : '{question}'  \n 답변 : \n ① **정의**: RAG(Retrieval-Augmented Generation, 검색 증강 생성)는 대규모 언어 모델(LLM)이 답변을 생성하기 전에 외부 지식 베이스나 문서 저장소에서 관련 정보를 먼저 검색(Retrieval)하고, 그 결과를 프롬프트에 결합해 정확한 답변을 생성(Generation)하도록 만든 AI 아키텍처입니다.\n② **이유(중요성)**: 기존 LLM은 학습 시점 이후의 최신 정보를 알지 못하거나, 모르는 내용을 사실처럼 지어내는 환각(Hallucination) 현상을 일으킬 수 있습니다. RAG 구조를 적용하면 모델을 매번 재학습시키지 않고도 최신 데이터나 사내 비공개 문서를 근거로 신뢰성 높은 답변을 제공할 수 있습니다.\n③ **쉬운 예시**: RAG는 시험을 볼 때 '오픈북 테스트'를 치르는 학생에 비유할 수 있습니다. 일반 LLM이 오직 머릿속에 외운 기억(사전 학습 데이터)만으로 문제를 푸는 학생이라면, RAG는 문제(사용자 질문)를 읽은 뒤 책상 위 전공 서적(외부 데이터베이스)에서 해당 페이지를 먼저 찾아보고 그 내용을 바탕으로 답안지(최종 답변)를 작성하는 것과 같습니다."),

    # [새로운 요청] — 실행 시 전달될 질문 변수. 요약 + 예시 + 비유 설명을 포함하여 대답해야 함
    ("user", "{question}")])

# 출력 파서 (응답 텍스트만 추출)
parser = StrOutputParser()

# LCEL 파이프라인 구성 (Prompt → LLM → Parser)
chain = prompt | llm | parser

# 실행 — invoke()에 직접 새로운 질문 전달
result = chain.invoke({"question": "REST API란?"})

print(result)


질문 : 'REST API란?'

답변 :
① 정의: REST API는 Representational State Transfer 원칙에 따라 웹에서 서로 다른 프로그램이 데이터를 주고받도록 하는 인터페이스입니다. 자원(Resource)은 URI로 식별하고, HTTP의 GET/POST/PUT/PATCH/DELETE 같은 메서드로 자원에 대한 작업을 수행합니다. 응답은 주로 JSON이나 XML 같은 표현으로 전달되며, 서버는 상태를 저장하지 않는 Stateless 방식으로 동작합니다.

② 이유(중요성): 표준화된 방식으로 시스템 간 소통이 쉬워지며, 언어와 플랫폼에 독립적입니다. 확장성·유연성, 캐싱 가능성, 다양한 서비스의 통합에 적합합니다.

③ 쉬운 예시: 소셜 미일 서비스의 REST 예
- GET /users/42 → 사용자 42의 프로필 조회
- POST /posts → 새 글 작성
- PUT /posts/123 → 글 123 수정
- DELETE /posts/123 → 글 123 삭제

④ 비유: 레스토랑 주문 비유
- 메뉴(URI)를 보고 원하는 음식(자원)을 선택
- 주문(HTTP 메서드)을 제출
- 주방(서버)이 주문을 처리해 결과를 서빙(응답)하는 흐름
- 각 요청은 독립적으로 처리되며, 같은 규칙으로 작동합니다.


<details>
<summary>정답 보기</summary>

```python 
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7,  max_tokens=200)
# gpt-5 같은 생각하는 모델은 tokens를 높게 잡지 않으면 생각단계에서 잘림 답변 안나올 수 있음

# 프롬프트 설계
# system: 챗봇의 역할 정의
# user: {question} 변수를 통해 질문 전달
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "너는 기술 개념을 알기 쉽게 설명하는 AI 선생님이야. "
     "답변은 반드시 ① 정의 ② 이유(중요성) ③ 쉬운 예시를 포함해야 해. "
     "필요하다면 비유를 사용해도 좋아."),
    ("user", "{question}")
])

# 출력 파서
parser = StrOutputParser()

# LCEL 체인 구성
chain = prompt | llm | parser

# 실행 테스트
question = "REST API란?"
answer = chain.invoke({"question": question})

print(f"🧠 질문: {question}\n")
print(f"💬 답변:\n{answer}")
```
</details>

##  5. 대화형 챗봇 (메모리)

### 🔹 Conversation Memory란?
> **Conversation Memory(대화 메모리)** 는 LLM이 **이전 대화 내용을 기억하고 이어서 대답**할 수 있도록 하는 기능입니다.

일반적인 LLM은 한 번의 요청(prompt)만 처리하고 끝나기 때문에,  
사용자가 “아까 말한 여행지 일정 다시 알려줘.”처럼 과거 대화를 참조하면 맥락을 잃어버립니다.  
→ 이때 필요한 것이 바로 **Memory(기억 기능)** 입니다.

### 🔹 왜 Memory가 필요할까?

| 상황 | Memory 없을 때 | Memory 있을 때 |
|------|----------------|----------------|
| 사용자: “부산 여행지 추천해줘.”<br>→ 모델이 답변함 | 다음 질문: “그럼 거기 근처 맛집은?”<br>→ 모델이 “어디 여행 말씀인가요?” | “부산 근처엔 회센터가 많고… 홍합탕집이 유명해요!” |
| 사용자: “어제 추천해준 책 제목 다시 말해줘.” | “어떤 책을 말씀하시는 건가요?” | “어제 말씀드린 건 『데미안』이에요.” |

👉 메모리를 활용하면,  
모델이 “과거 대화의 맥락(Context)”을 유지한 채로  
**자연스러운 멀티턴(Multi-turn) 대화형 챗봇**을 만들 수 있습니다.

### 🔹 LangChain에서의 Memory 개념

LangChain은 다양한 형태의 “기억 클래스(memory classes)”를 제공합니다.  
대표적으로 아래 세 가지를 이해하면 충분합니다 👇

| Memory 클래스 | 설명 | 특징 |
|----------------|------|------|
| **`ConversationBufferMemory`** | 단순히 대화 전체를 계속 저장 | 가장 기본적이고 직관적 |
| **`ConversationBufferWindowMemory`** | 최근 N개의 대화만 저장 | “단기 기억” 형태로, 긴 대화를 효율적으로 유지 |
| **`ConversationSummaryMemory`** | 이전 대화를 LLM이 요약해서 저장 | 긴 대화를 핵심 요약본으로 관리, 맥락 유지에 적합 |
    #-- 대화용, 비서 챗봇 등 (요약된 대화 + 현재 질문)

### 🔹 프롬프트에 직접 넣는 방식 vs Memory 클래스 사용 비교

| 비교 항목 | 프롬프트에 직접 삽입 | Memory 클래스 사용 |
|------------|----------------------|---------------------|
| 코드 복잡도 | ❌ 대화마다 프롬프트를 새로 생성해야 함 | ✅ LangChain이 자동으로 이전 대화 삽입 |
| 확장성 | ❌ 대화가 길어지면 토큰 초과 문제 발생 | ✅ 오래된 대화는 요약/제한 가능 |
| 유지보수 | ⚠️ 과거 대화 삽입 위치만 교체하면 되지만, 요약·토큰 제어를 직접 구현해야 함 | ✅ 메모리 객체만 관리하면 됨 |
| 현실성 | ❌ “일회용 챗봇”에만 적합 | ✅ “지속형 대화형 챗봇” 구현 가능 |

즉, **Memory는 LLM이 “대화 히스토리를 스스로 관리”하도록 하는 스마트한 도우미 클래스**입니다.

### 🔹 Memory 작동 원리
1. 사용자가 LLM에 **질문(입력)** 을 보냅니다.  

2. LLM이 **답변**을 생성하면, 이 대화 내용이 자동으로 **Memory에 저장**됩니다.  
3. 사용자가 다음 질문을 입력하면,  
   LangChain이 **이전 대화 기록을 자동으로 프롬프트에 포함시켜** LLM에 전달합니다.  
4. LLM은 이 정보를 바탕으로 **맥락을 이해하고 연속적인 대화**를 생성합니다.

→ 즉, 사용자는 별도로 과거 대화를 넣을 필요 없이, LangChain이 **“대화 기록 → 프롬프트 삽입 → LLM 호출”**  
과정을 자동으로 처리합니다.

### 코드 예시 1. 기본 Buffer Memory (대화 전체 저장)

In [ ]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferMemory(return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
    MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
    ("human", "{input}")                           # 사용자 질문 (사람지침 human / man / user)
]) 

# LCEL 체인 구성
chain = prompt | llm 

# 5️⃣ 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "그럼 그 근처 맛집은 어디야?"]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})

모델은 앞선 “부산 여행” 대화를 기억하고 자연스럽게 이어서 대답합니다.

In [ ]:
memory.load_memory_variables({})['history']

### 코드 예시 2. 최근 대화만 유지 (Window Memory)

In [ ]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferWindowMemory(k=2, return_messages=True)
# k=2 --> 기억력 설정: 과거 대화 최근 순으로 몇개 저장하는지 입력값
# 요약 기능은 gpt-4-mini 추천 / 답변 기능은 gpt-5
# k는 단기메모리 장기기억은 별도의 요약파일로 관리

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
    MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
    ("human", "{input}")                           # 사용자 질문
]) 

# LCEL 체인 구성
chain = prompt | llm 

# 5️⃣ 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "대한 민국의 수도는 어디야?", "서울의 인구는 몇명이야?", "내가 아까 추천해달라고 한 어행지는 어디야?" ]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})



사용자: 부산 여행지 추천해줘.
 응답: 멋진 선택이에요! 부산은 바다를 따라 걷는 것도, 맛집 탐방도, 문화 체험도 가능한 도시예요. 분위기별로 간단하게 추천드릴게요. 취향이나 여행 기간 알려주시면 더 맞춤 추천도 드릴게요.

1) 꼭 가봐야 할 부산 명소
- 해운대 해수욕장 + 동백섬 산책로
  - 왜 좋나요: 넓은 모래사장, 맑은 바다, 아름다운 산책로. 해질녘이 특히 멋져요.
  - 팁: 해수욕 시즌이 아니더라도 해운대 해변 산책로와 동백섬 데크길이 좋습니다. 근처에 카페나 해산물 맛집 많음.
- 광안리 해수욕장 + 광안대교 야경
  - 왜 좋나요: 해변 분위기 좋고, 광안대교의 멋진 야경 포인트.
  - 팁: 해질 무렵 산책 후 광안대교 다리 아래에서 야경 감상 추천.
- 남포동, BIFF Square, 자갈치시장
  - 왜 좋나요: 영화의 거리 분위기, 해산물 시장 체험, 길거리 간식 즐기기.
  - 팁: 자갈치시장 상점가에서 신선한 해산물 구매 후 바로 조리해 주는 식당도 많아요.
- 감천문화마을
  - 왜 좋나요: 알록달록한 계단식 마을과 벽화, 전망 포인트가 많아 인생 사진 잘 나와요.
  - 팁: 골목 골목의 작은 카페도 많아 느긋하게 걷기 좋습니다.
- 태종대 공원
  - 왜 좋나요: 바다 절벽 풍경과 등대 전망이 멋진 자연 명소.
  - 팁: 산책로에 경사로가 있어 체력에 맞춰 구간 선택이 필요합니다.
- 해동용궁사
  - 왜 좋나요: 바다를 바라보는 사찰로 운치 있는 분위기.
  - 팁: 바닷바람이 셀 수 있으니 겉옷 챙기기.
- 오륙도 스카이워크
  - 왜 좋나요: 바다를 360도 조망하는 스카이워크와 전망대.
  - 팁: 바람이 강할 때는 안전에 주의, 날씨 좋은 날 조난 포인트에서 사진 찍기 좋음.
- 송도해수욕장(송도구) 및 근교 산책
  - 왜 좋나요: 비교적 한적한 해변 분위기, 가족 단위 여행에 좋음.
- 부산타워(용두산공원)
  - 왜 좋나요: 시내 전망 포인트, 남포동 일대 쉽게 접근 가능.
- 흰여울문화마을(영도 주변

이 방식은 단기 기억처럼 작동합니다.  
오래된 대화는 자동으로 지워져 토큰 낭비를 줄이고 효율성을 높입니다.

### 코드 예시 3. 이전 대화를 요약해서 저장 (ConversationSummaryMemory)

In [23]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 요약형 메모리 생성 — LLM이 과거 대화를 요약해 맥락을 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 플래너야. 사용자의 요구에 따라 가족 여행 일정을 제안해줘."),
    MessagesPlaceholder(variable_name="history"),  # 요약된 과거 대화 자동 삽입
    ("human", "{input}")                           # 현재 질문
])

# LCEL 체인 구성
chain = prompt | llm

# 연속 대화 시뮬레이션
inputs = [
    "이번 주말에 가족 여행지 추천해줘.",
    "지난번에 추천한 곳 중에 아이들이 놀기 좋은 곳은 어디였지?",
    "그럼 거기 일정표를 하루만 짜줘."
]

for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n👤 사용자: {user_input}\n🤖 응답: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

C:\Users\HPE\AppData\Local\Temp\ipykernel_14768\2842420670.py:9: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, return_messages=True)



👤 사용자: 이번 주말에 가족 여행지 추천해줘.
🤖 응답: 좋아요! 이번 주말 가족 여행을 몇 가지 테마로 바로 제안드릴게요. 출발지나 가족 구성(아이 나이) 등 정보를 주시면 더 맞춤으로 다듬어 드릴 수 있어요. 우선 당일치기와 1박2일로 활용하기 좋은 아이디어들입니다.

1) 가까운 자연 속 당일치기 힐링
- 어디가 좋나요? 서울/수도권 근처의 가평, 양평, 청평권 등
- 왜 좋냐? 차로 1~2시간 정도로 가볍게 다녀올 수 있고, 아이와 함께 자연 체험이 가능해 피곤하지 않으면서도 재충전 가능
- 예시 일정(당일치기):
  - 아침 출발 → 자라섬/두물머리 산책 → 점심 피크닉 또는 현지 맛집 → 강가나 산책로 산책 마무리 → 오후 귀가
- 팁: 주말 혼잡 시간 피하기, 주차장 위치 확인, 아이들 물놀이 용품 준비

2) 가족 테마파크 당일치기
- 추천 장소: 에버랜드(용인) 또는 롯데월드(잠실) 등
- 왜 좋냐? 연령대가 다양해도 함께 즐길 수 있는 어트랙션이 많고, 퍼레이드/쇼도 있어 가족끼리 즐겁습니다
- 예시 일정(당일치기):
  - 이른 시간 입장권 구입 → 인기 놀이기구 우선 타기 → 점심식사 → 동물원/숲 속 산책 → 불꽃/퍼레이드 관람(시즌에 따라 다름) → 저녁 귀가
- 팁: 파크 조기 도착, Fast Pass/예약권 활용, 물과 간식 준비

3) 해변·바다 근교 주말 1박2일
- 후보지: 동해안(속초, 강릉 등)이나 남해권 근처
- 왜 좋냐? 바다를 보며 여유 있게 쉬고, 아이들이 해양 체험학습 같은 활동도 가능
- 예시 일정(1박2일, 서울/수도권 출발 기준):
  - 금요일 저녁 출발 → 숙소 체크인 → 해변 산책/해수욕(계절에 따라) → 다음날 아침 해돋이 감상 → 수족관/해변 근처 산책 → 점심 후 귀가
- 팁: 주말 도로 혼잡 대비, 미리 숙소 예약 필수, 날씨 체크

4) 문화·역사 체험 코스
- 후보지: 수원 화성, 안성 도자기 마을, 전주 한옥마을(멀지만 멀지 않다면 1박 필수)
- 왜 좋냐? 아이들에게 역사와 지역 문화를

In [24]:
# 현재까지의 요약본 확인
print(memory.buffer)

New summary: 인간은 다섯 가지 테마 중 아이들이 놀기에 특히 좋은 곳이 어디인지 물었고, AI는 2) 가족 테마파크 당일치기, 3) 해변·바다 근교 주말 1박2일, 5) 실내·과학·체험형이 특히 적합하다고 답했다. 또한 아이 연령에 따라 포인트가 달라지는 1) 가까운 자연 속 당일치기, 4) 문화·역사 체험 코스도 함께 설명했다. 각 테마에 대해 추천 이유와 아이 연령에 따른 포인트, 간단 예시 일정이 제시되었고, 이후 원한다면 아이의 연령대, 출발지, 예산, 이동 수단을 알려주면 1) 구체적 장소 2) 당일/1박2일 일정 3) 예상 소요 시간과 동선을 포함한 맞춤 일정표로 바로 만들어 드리겠다고 했다. 그다음 인간이 '그럼 거기 일정표를 하루만 짜줘'라고 요청하자 AI는 '좋아요. 바로 사용할 수 있는 한 가지 하루치 일정 예시를 먼저 드리고, 원하시면 아이 연령대, 출발지, 예산, 이동 수단 등에 맞춰 바로 맞춤안으로 바꿔드릴게요'라고 답했다. 이어 예시 1안으로 실내·과학·체험형 당일치기(서울/경기 근교, 아이 연령대 4–9세 가정에 적합)가 제시되었고, 일정 요약, 소요 시간, 추천 장소 유형, 상세 일정표, 아이 연령별 포인트, 예산 관리, 동선 팁, 준비물 체크리스트가 포함된 포괄적인 일정이 제시되었다. 또한 원하시면 위의 1안과 동일한 형식의 2안(다른 장소/테마)도 아이 연령대와 출발지에 맞춰 바로 만들어 드리겠다고 했다.


이 방식은 LLM이 이전 대화를 스스로 요약하여 핵심 맥락을 유지합니다.  
수백 문장의 대화도 핵심 내용만 남기기 때문에 효율적입니다.

**핵심 정리**
| Memory 타입 | 특징 | 사용 목적 |
|--------------|--------|-------------|
| **BufferMemory** | 모든 대화를 그대로 저장 | 짧은 대화, 디버깅용 |
| **WindowMemory** | 최근 N개의 대화만 유지 | 실시간 채팅, 효율성 |
| **SummaryMemory** | 이전 대화를 요약해서 유지 | 긴 대화 맥락 유지형 챗봇 |

🔹 실습 : 아래 조건을 만족하는 “나만의 대화형 여행 상담 챗봇”을 만들어보세요.

당신은 AI 여행 비서 트래블GPT 입니다.  
고객이 여러 도시를 순서대로 여행하면서 맞춤 일정·숙소·음식을 요청할 때,  
이전 대화 내용을 기억하며 연결된 제안을 해주는 지능형 여행 어시스턴트를 구현하세요.  

1. ChatOpenAI(model="gpt-5-nano") 사용
2. ConversationSummaryMemory로 요약 기반 대화 기억 구현
3. 답변에 반드시 "이전 여행 내용을 바탕으로 추천드리면..." 이라는 문구 포함
  - 예시 : 이전 여행 내용을 바탕으로 추천드리면, 여수에서는 해상케이블카와 낭만포차거리를 꼭 가보세요.
4. 대화 시나리오:
  - 사용자가 “부산 → 여수 → 강릉” 순으로 도시를 이동
  - 챗봇은 이전 도시에서 한 활동을 기억하고 “연결된 여행 루트”나 “테마별 추천(가족/커플/힐링)”을 제안할 것

예시 시나리오
```bash
사용자: 이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.
AI: 부산의 해운대, 아쿠아리움이 가족 단위로 인기예요!

사용자: 이번엔 여수로 가볼까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 부산의 해변 감성에 이어 여수에서는 바다 전망 케이블카와 낭만포차를 즐기세요.

사용자: 그럼 마지막은 강릉이 좋을까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 강릉에서는 여수보다 조용한 힐링 카페 거리와 바다 일출 코스를 권합니다.
```

In [25]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 1. 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 2. 요약형 메모리 생성
memory = ConversationSummaryMemory(
    # TODO: 필요한 인자를 채워보세요.
    llm=llm, return_messages=True
)
 
# 3. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     # TODO: 트래블GPT의 역할과 반드시 포함할 문구 조건을 작성해보세요.
     "너는 여행 비서 트래블 GPT 야. "
     "사용자의 여행 루트를 기억하고, 이전 여행 내용을 바탕으로 다음 여행할 도시를 추천해줘. "
     "답변에는 반드시 '이전 여행 내용을 바탕으로 추천드리면,' 이라는 문구를 포함해야 해."
    ),
    # TODO: 이전 대화(history)가 들어갈 자리를 추가해보세요.
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 4. LCEL 체인 구성
chain = prompt | llm

# 5. 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 6. 연속 대화 실행
for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

# 7. 선택: 요약된 메모리 확인
# print("\n요약된 Memory Buffer:")
print(memory.buffer)


사용자: 이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.
트래블GPT: 이전 여행 내용을 바탕으로 추천드리면, 이번 주말 부산 가족 여행에 딱 맞는 장소를 엄선해봤어요.

가족 친화적 부산 명소 6선
- 해운대 해수욕장 + 동백섬 산책
  왜 좋은가: 넓은 모래사장에서 놀고, 동백섬 산책로를 따라 바다 풍경도 감상하기 좋아요. 아이들도 비교적 안전하게 걷기 좋고, 근처에 맛집과 카페도 많아요.
  팁: 주차가 어려울 수 있어 지하철 이용을 추천. 해가 질 때 동백섬에서 찍는 사진이 아주 예뻐요.

- 부산 아쿠아리움(Sea Life Busan Aquarium)
  왜 좋은가: 다양한 해양 생물을 한자리에서 볼 수 있어 아이들 관심이 오래 mantenerse. 실내라 비 오는 날에도 좋습니다.
  팁: 해운대 근처에 위치해 해운대 일정과 함께 묶기 쉬워요. 2–3시간 정도 여유 두고 방문 권장.

- 태종대 공원 및 유람선
  왜 좋은가: 멋진 해안 절경과 등대 구간이 아이들에게도 모험심을 자극합니다. 해안 산책로가 잘 정비되어 있어 가족 단위 방문에 적합.
  팁: 날씨가 맑은 날에 유람선 시간대를 확인해 미리 예약. 바람이 많이 부는 날은 방풍옷 준비.

- 감천문화마을
  왜 좋은가: 알록달록한 집들이 포토 스폿으로 유명하고 산책로를 따라 다양한 벽화와 조형물을 즐길 수 있습니다.
  팁: 계단이 많아 아이들과 함께 걷기 편한 동선을 미리 파악하면 좋습니다. 엘리베이터 위치를 확인해 보세요.

- 광안리 해변 + 광안대교 야경
  왜 좋은가: 해변 산책로가 넓고, 저녁엔 광안대교의 빛나는 야경이 멋져요. 가족 피크닉이나 가볍게 산책하기 좋습니다.
  팁: 해질 무렵이 가장 아름다운 시간대예요. 아이와 함께 모래놀이도 좋고 근처 해산물 eateries 도 많아요.

- 오륙도 스카이워크
  왜 좋은가: 바다 위의 유리 바닥 플랫폼에서 색다른 시야를 경험할 수 있습니다. 비교적 짧은 코스라 아이들이 체력 부담이 적어요.
  팁: 다리길이가 짧지만 안전장

<details>
<summary>정답 보기</summary>

```python
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 메모리 생성 — 이전 대화를 요약하며 맥락 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 여행 비서 트래블GPT야. "
     "사용자의 여행 루트를 기억하고, 이전 여행 내용을 바탕으로 다음 도시를 추천해줘. "
     "답변에는 반드시 '이전 여행 내용을 바탕으로 추천드리면,' 이라는 문구를 포함해야 해."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# LCEL 체인 구성
chain = prompt | llm

# 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 연속 대화 시뮬레이션
for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

# 대화 요약 확인 (선택)
print("\n요약된 Memory Buffer:")
print(memory.buffer)
```
</details>